In [ ]:
# ══════════════════════════════════════════════
# PRELUDE — run this cell first
# Added by fix pass. Everything below your own code is unchanged.
# ══════════════════════════════════════════════

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timezone

class Stale(RuntimeError): pass
class Unconverged(RuntimeError): pass
class TooFewObs(RuntimeError): pass


def safe_at(obj, i=-1, col=None):
    """float() on a 1-element Series is deprecated. Handles yfinance MultiIndex columns."""
    s = obj[col] if col is not None else obj
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    a = np.asarray(s.dropna()).ravel()
    if a.size == 0:
        raise Stale("empty series")
    return float(a[i])


def safe_last(obj, col=None):
    return safe_at(obj, -1, col)


def unmute_convergence():
    """Let convergence failures through. They were being swallowed by filterwarnings('ignore')."""
    try:
        import statsmodels.tools.sm_exceptions as _s
        for _n in ("ConvergenceWarning", "EstimationWarning", "ValueWarning"):
            if hasattr(_s, _n):
                warnings.filterwarnings("always", category=getattr(_s, _n))
    except Exception:
        pass
    try:
        from arch.utility.exceptions import DataScaleWarning
        warnings.filterwarnings("always", category=DataScaleWarning)
    except Exception:
        pass


# ── CONTRACT: pin which gold series this notebook uses ──────────────
#   'front_intraday' = GC=F 30-min  (the contract you trade — DEFAULT)
#   'front_daily'    = GC=F daily   (often prints spot, not the future)
#   'spot'           = XAUUSD
LIVE_CONTRACT = "front_intraday"


def get_ctx(contract=LIVE_CONTRACT):
    if contract == "front_intraday":
        gc_df = yf.download("GC=F", period="5d", interval="30m", progress=False)
    elif contract == "front_daily":
        gc_df = yf.download("GC=F", period="1mo", interval="1d", progress=False)
    elif contract == "spot":
        gc_df = yf.download("XAUUSD=X", period="5d", interval="30m", progress=False)
    else:
        raise ValueError(contract)

    gc = safe_last(gc_df, "Close")
    gld = safe_last(yf.download("GLD", period="5d", progress=False), "Close")

    def _s(t, d):
        try:
            return safe_last(yf.download(t, period="5d", progress=False), "Close")
        except Exception:
            return d

    if not 1000 < gc < 12000:
        raise Stale(f"GC={gc:,.2f} implausible — bad fetch")
    if not 100 < gld < 1200:
        raise Stale(f"GLD={gld:,.2f} implausible — bad fetch")
    ratio = gc / gld
    if not 9.5 <= ratio <= 12.5:
        raise Stale(f"GLD->gold ratio {ratio:.3f}x outside [9.5, 12.5] — "
                    f"prices are from different dates")

    # known failure point: GC=F daily and 30m disagree
    try:
        _d = safe_last(yf.download("GC=F", period="1mo", interval="1d", progress=False), "Close")
        _i = safe_last(yf.download("GC=F", period="5d", interval="30m", progress=False), "Close")
        if abs(_d - _i) / _i > 0.005:
            print(f"  !! GC=F daily {_d:,.2f} vs 30m {_i:,.2f} — ${abs(_d-_i):,.1f} apart.")
            print(f"     Using '{contract}'. VERIFY THE SETTLE IN QUANTOWER.")
    except Exception:
        pass

    ctx = dict(gc=gc, gld=gld, ratio=ratio, contract=contract,
               asof=str(gc_df.index[-1]),
               vix=_s("^VIX", np.nan), gvz=_s("^GVZ", np.nan), rf=_s("^IRX", 4.0) / 100)
    print(f"  contract : {contract}")
    print(f"  GC {gc:>10,.2f}   GLD {gld:>8,.2f}   ratio {ratio:.4f}x   <-- NOT 10.0")
    print(f"  VIX {ctx['vix']:.2f}   GVZ {ctx['gvz']:.1f}   rf {ctx['rf']:.2%}   as of {ctx['asof']}")
    return ctx


def need_obs(n, floor, label=""):
    if n < floor:
        raise TooFewObs(f"{label}: {n} observations, need >= {floor}. Do not report this fit.")


def need_fresh(as_of, days=7, label="field"):
    age = (datetime.now(timezone.utc).date() - datetime.fromisoformat(as_of).date()).days
    if age > days:
        raise Stale(f"{label} written {as_of} ({age}d ago) — EXPIRED. Rewrite or delete it.")
    return True


def need_converged(res, states=None, label="model"):
    conv = getattr(res, "converged", None)
    if conv is None and isinstance(getattr(res, "mle_retvals", None), dict):
        conv = res.mle_retvals.get("converged", True)
    if conv is False:
        raise Unconverged(f"{label}: optimiser did not converge. Not a regime classification.")
    if states is not None:
        u, c = np.unique(np.asarray(states), return_counts=True)
        if len(u) < 2:
            raise Unconverged(f"{label}: {len(u)} distinct state — a flat line, not regimes.")
        if c.min() / c.sum() < 0.05:
            raise Unconverged(f"{label}: minority state is {c.min()/c.sum():.1%} — "
                              f"outlier detector, not a regime model.")


def kelly_cap(f, frac=0.25, cap=0.20):
    out = float(np.clip(f * frac, -cap, cap))
    if abs(f) > 1:
        print(f"  ! raw f*={f:.2f} implies {f*100:.0f}% of capital (small-sample artefact). "
              f"Using {out:.1%}.")
    return out


LIVE_CTX   = get_ctx()
LIVE_RATIO = LIVE_CTX["ratio"]   # use instead of 10
LIVE_GC    = LIVE_CTX["gc"]

# NOTE: named LIVE_* on purpose — GOLD is your hex colour in 15 cells,
#       and RATIO / CONTRACT are already used in cells 44-45.


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# COT AUTO-FETCH — CFTC Disaggregated, Futures-and-Options COMBINED
# Source: https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm
#
# NOTE: your old code pulled f_year.txt from fut_disagg_* = FUTURES ONLY,
#       while every dashboard header said "Options and Futures Combined".
#       This uses com_disagg_* -> c_year.txt = actually COMBINED.
#
# The annual zip is rewritten by CFTC every Friday ~15:30 ET, so pulling
# the current year's file always gives the newest report. No manual step.
# ══════════════════════════════════════════════════════════════════════

import io, os, zipfile, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path

COT_CACHE = Path.home() / ".cot_cache"
COT_CACHE.mkdir(exist_ok=True)

# Disaggregated Futures-and-Options Combined, annual history
COT_HIST_URL = "https://www.cftc.gov/files/dea/history/com_disagg_txt_{year}.zip"
# Current-week snapshot (headerless; used only as a freshness cross-check)
COT_CURRENT_URL = "https://www.cftc.gov/dea/newcot/c_disagg.txt"

CFTC_CODES = {
    "gold":        "088691",
    "silver":      "084691",
    "copper":      "085692",
    "wti_crude":   "067651",
    "natgas":      "023651",
    "corn":        "002602",
    "platinum":    "076651",
    "palladium":   "075651",
}

_COT_UA = {"User-Agent": "Mozilla/5.0 (research; contact: elena)"}


def _cot_expected_report_date(now=None):
    """
    COT is Tuesday data released Friday 15:30 ET. Returns the latest
    Tuesday that should be published by now.
    """
    now = now or datetime.now(timezone.utc)
    et = now - timedelta(hours=4)                    # ET approx (EDT)
    days_since_fri = (et.weekday() - 4) % 7
    last_fri = (et - timedelta(days=days_since_fri)).replace(
        hour=15, minute=30, second=0, microsecond=0)
    if et < last_fri:
        last_fri -= timedelta(days=7)
    return (last_fri - timedelta(days=3)).date()     # the Tuesday it covers


def _cot_download_year(year, force=False):
    """Download and cache one year of combined disaggregated data."""
    cache = COT_CACHE / f"com_disagg_{year}.parquet"
    stamp = COT_CACHE / f"com_disagg_{year}.stamp"

    # current year: refresh if cache older than 12h; past years never change
    fresh = False
    if cache.exists() and not force:
        if year < datetime.now().year:
            fresh = True
        elif stamp.exists():
            age_h = (datetime.now().timestamp() - float(stamp.read_text())) / 3600
            fresh = age_h < 12
    if fresh:
        return pd.read_parquet(cache)

    url = COT_HIST_URL.format(year=year)
    r = requests.get(url, timeout=60, headers=_COT_UA)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        names = z.namelist()
        inner = next((n for n in names if n.lower().endswith((".txt", ".csv"))), None)
        if inner is None:
            raise RuntimeError(f"{year}: no txt/csv in zip — got {names}")
        if inner.lower().startswith("f_"):
            raise RuntimeError(
                f"{year}: zip contains '{inner}' (f_ = FUTURES ONLY). "
                f"Expected 'c_year.txt' from com_disagg_*. Wrong URL.")
        with z.open(inner) as fh:
            df = pd.read_csv(fh, low_memory=False)

    df.columns = [c.strip() for c in df.columns]
    cache.parent.mkdir(exist_ok=True)
    df.to_parquet(cache, index=False)
    stamp.write_text(str(datetime.now().timestamp()))
    print(f"    downloaded {year}  ({inner}, {len(df):,} rows)")
    return df


def _cot_date_col(df):
    for c in ("Report_Date_as_YYYY-MM-DD", "Report_Date_as_MM_DD_YYYY",
              "As_of_Date_In_Form_YYMMDD", "Report_Date"):
        if c in df.columns:
            return c
    raise KeyError(f"no date column found in {list(df.columns)[:12]}")


def _cot_parse_dates(s, col):
    if "YYMMDD" in col:
        return pd.to_datetime(s.astype(str).str.zfill(6), format="%y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")


def fetch_cot(market="gold", years=3, force=False, verbose=True):
    """
    Returns a tidy weekly DataFrame for one market, newest last:

        date, open_interest,
        prod_long, prod_short, prod_net,
        swap_long, swap_short, swap_net,
        mm_long,   mm_short,   mm_net,
        other_net, comm_net (prod+swap),
        mm_index, comm_index   (156-week rolling 0-100 Williams index)

    Raises on anything implausible rather than returning zeros.
    """
    code = CFTC_CODES.get(market.lower(), market)
    this_year = datetime.now().year
    frames = []
    if verbose:
        print(f"  CFTC Disaggregated · Futures-and-Options COMBINED · code {code}")

    for y in range(this_year - years + 1, this_year + 1):
        try:
            frames.append(_cot_download_year(y, force=force))
        except Exception as e:
            print(f"    ! {y}: {type(e).__name__}: {e}")
    if not frames:
        raise RuntimeError("no COT data retrieved — check network / CFTC availability")

    raw = pd.concat(frames, ignore_index=True)

    code_col = next((c for c in ("CFTC_Contract_Market_Code", "CFTC_Contract_Market_Code_Quotes")
                     if c in raw.columns), None)
    if code_col is None:
        raise KeyError("no CFTC_Contract_Market_Code column")

    m = raw[raw[code_col].astype(str).str.strip().str.zfill(6) == code].copy()
    if m.empty:
        names = raw["Market_and_Exchange_Names"].dropna().unique()[:5]
        raise ValueError(f"code {code} not found. Sample markets: {list(names)}")

    dcol = _cot_date_col(m)
    m["date"] = _cot_parse_dates(m[dcol], dcol)
    m = m.dropna(subset=["date"]).sort_values("date")
    m = m.drop_duplicates(subset=["date"], keep="last")

    def col(*cands):
        for c in cands:
            if c in m.columns:
                return pd.to_numeric(m[c], errors="coerce")
        raise KeyError(f"none of {cands} present")

    out = pd.DataFrame({
        "date":          m["date"].values,
        "open_interest": col("Open_Interest_All"),
        "prod_long":     col("Prod_Merc_Positions_Long_All"),
        "prod_short":    col("Prod_Merc_Positions_Short_All"),
        "swap_long":     col("Swap_Positions_Long_All"),
        "swap_short":    col("Swap__Positions_Short_All", "Swap_Positions_Short_All"),
        "mm_long":       col("M_Money_Positions_Long_All"),
        "mm_short":      col("M_Money_Positions_Short_All"),
        "other_long":    col("Other_Rept_Positions_Long_All"),
        "other_short":   col("Other_Rept_Positions_Short_All"),
    }).reset_index(drop=True)

    out["prod_net"]  = out.prod_long  - out.prod_short
    out["swap_net"]  = out.swap_long  - out.swap_short
    out["mm_net"]    = out.mm_long    - out.mm_short
    out["other_net"] = out.other_long - out.other_short
    out["comm_net"]  = out.prod_net + out.swap_net     # producers + swap dealers

    w = min(156, len(out))
    for c in ("mm_net", "comm_net", "swap_net"):
        lo = out[c].rolling(w, min_periods=20).min()
        hi = out[c].rolling(w, min_periods=20).max()
        out[c.replace("_net", "_index")] = 100 * (out[c] - lo) / (hi - lo).replace(0, np.nan)

    # ---- validation: fail loud rather than score a broken parse ----------
    last = out.iloc[-1]
    if last.open_interest < 10_000:
        raise ValueError(f"open interest {last.open_interest:,.0f} implausible — bad parse")
    for f in ("prod_net", "swap_net", "mm_net"):
        if last[f] == 0:
            raise ValueError(f"{f} parsed as exactly 0 — column mismatch, not flat positioning")

    expected = _cot_expected_report_date()
    lag = (expected - last.date.date()).days
    if lag > 7:
        print(f"    !! newest report {last.date.date()} but {expected} expected "
              f"({lag}d stale). CFTC may be delayed, or cache is stale — force=True to refresh.")
    elif verbose:
        print(f"    latest report : {last.date.date()}  (current)")

    if verbose:
        print(f"    rows          : {len(out)}  [{out.date.min().date()} -> {out.date.max().date()}]")
        print(f"    OI            : {last.open_interest:>10,.0f}")
        print(f"    Managed Money : {last.mm_net:>+10,.0f}   index {last.mm_index:5.1f}/100")
        print(f"    Swap Dealers  : {last.swap_net:>+10,.0f}   index {last.swap_index:5.1f}/100")
        print(f"    Prod/Merch    : {last.prod_net:>+10,.0f}")
        print(f"    Commercial    : {last.comm_net:>+10,.0f}   index {last.comm_index:5.1f}/100")

    return out


def cot_signal(df, extreme_lo=20, extreme_hi=80):
    """Williams rule: need BOTH commercial and managed-money at an extreme."""
    r = df.iloc[-1]
    c, m = r.comm_index, r.mm_index
    if np.isnan(c) or np.isnan(m):
        return "INSUFFICIENT HISTORY", 0
    if c >= extreme_hi and m <= extreme_lo:
        return "BULLISH — commercials long, specs washed out", +1
    if c <= extreme_lo and m >= extreme_hi:
        return "BEARISH — commercials short, specs crowded", -1
    return f"NEUTRAL — no extreme (comm {c:.0f}, mm {m:.0f})", 0


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt

# -----------------------------
# 1️⃣ Paste your COT report as a string
# -----------------------------
cot_text = """
GOLD - COMMODITY EXCHANGE INC.                                                                                                                   Code-088691
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 100 TROY OUNCES)                                                                                          :
     :          :    Positions                                                                                                           :
All  :   564,585:    21,252     47,000     17,640    242,776     65,403    151,491      9,623     36,914     96,479     23,491    117,723:   57,683    21,656
Old  :   564,585:    21,252     47,000     17,640    242,776     65,403    151,491      9,623     36,914     96,479     23,491    117,723:   57,683    21,656
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :    76,273:     1,487     10,832     -2,241     18,954      3,851      9,671        201      7,303     16,918      2,180     31,241:    8,044     1,712
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       3.8        8.3        3.1       43.0       11.6       26.8        1.7        6.5       17.1        4.2       20.9:     10.2       3.8
Old  :     100.0:       3.8        8.3        3.1       43.0       11.6       26.8        1.7        6.5       17.1        4.2       20.9:     10.2       3.8
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :       317:        22         20         12         30         35         97         17         68         97         38         67:
Old  :       317:        22         20         12         30         35         97         17         68         97         38         67:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 18.6       29.4       28.5       45.6       13.9       26.4       18.8       37.8
Old  :                 18.6       29.4       28.5       45.6       13.9       26.4       18.8       37.8
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
 
MICRO GOLD - COMMODITY EXCHANGE INC.                                                                                                             Code-088695
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 10 TROY OUNCES)                                                                                           :
     :          :    Positions                                                                                                           :
All  :    61,419:       242          0      6,755          0          0        577        202         64     13,601     28,913      5,825:   34,355    26,415
Old  :    61,419:       242          0      6,755          0          0        577        202         64     13,601     28,913      5,825:   34,355    26,415
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :     5,908:         0          0        -55          0          0       -554        202        -30      3,221      2,808         70:    3,256     2,858
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       0.4        0.0       11.0        0.0        0.0        0.9        0.3        0.1       22.1       47.1        9.5:     55.9      43.0
Old  :     100.0:       0.4        0.0       11.0        0.0        0.0        0.9        0.3        0.1       22.1       47.1        9.5:     55.9      43.0
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :        37:         .          0          .          0          0          .          .          .         16         13         13:
Old  :        37:         .          0          .          0          0          .          .          .         16         13         13:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 22.7       44.5       30.9       52.0       21.0       42.6       26.9       45.7
Old  :                 22.7       44.5       30.9       52.0       21.0       42.6       26.9       45.7
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
"""

# -----------------------------
# 2️⃣ Function to parse each section (Gold / Micro Gold)
# -----------------------------
def parse_cot_section(text, section_name):
    # Extract the section by name
    section_pattern = rf"{section_name}.*?All\s*:"
    section_match = re.search(rf"{section_name}.*?(?=MICRO GOLD|$)", text, re.DOTALL)
    if not section_match:
        print(f"⚠️ Section {section_name} not found!")
        return None
    
    section = section_match.group(0)
    
    # Grab all "All" lines with numbers
    pattern = r"All\s*:\s*([\d,]+):\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)"
    matches = re.findall(pattern, section)
    
    if not matches:
        print(f"⚠️ No 'All' data found in {section_name}")
        return None
    
    data = []
    for m in matches:
        data.append([int(x.replace(',', '')) for x in m])
    
    columns = [
        "Open_Interest",
        "NonCommercial_Long",
        "NonCommercial_Short",
        "NonCommercial_Spreading",
        "Commercial_Long",
        "Commercial_Short",
        "TotalReportable_Long",
        "TotalReportable_Short"
    ]
    
    df = pd.DataFrame(data, columns=columns)
    
    # Calculate Net Positions
    df['NonCommercial_Net'] = df['NonCommercial_Long'] - df['NonCommercial_Short']
    df['Commercial_Net'] = df['Commercial_Long'] - df['Commercial_Short']
    
    # Generate signals based on extremes (top/bottom 10% Non-Commercial Net)
    long_thresh = df['NonCommercial_Net'].quantile(0.1)
    short_thresh = df['NonCommercial_Net'].quantile(0.9)
    
    df['Signal'] = 0
    df.loc[df['NonCommercial_Net'] > short_thresh, 'Signal'] = -1  # likely sell
    df.loc[df['NonCommercial_Net'] < long_thresh, 'Signal'] = 1    # likely buy
    
    return df

# -----------------------------
# 3️⃣ Parse Gold & Micro Gold
# -----------------------------
gold_df = parse_cot_section(cot_text, "GOLD - COMMODITY EXCHANGE INC")
micro_df = parse_cot_section(cot_text, "MICRO GOLD - COMMODITY EXCHANGE INC")

print("✅ Gold COT Data:")
print(gold_df[['NonCommercial_Net','Commercial_Net','Signal']])

print("\n✅ Micro Gold COT Data:")
print(micro_df[['NonCommercial_Net','Commercial_Net','Signal']])

# -----------------------------
# 4️⃣ Plot Net Positions for visualization
# -----------------------------
plt.figure(figsize=(10,5))
plt.plot(gold_df['NonCommercial_Net'], label='Gold Non-Commercial Net', marker='o')
plt.plot(gold_df['Commercial_Net'], label='Gold Commercial Net', marker='x')
plt.title('Gold Futures COT Net Positions')
plt.xlabel('Entry (All Rows)')
plt.ylabel('Contracts')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10,5))
plt.plot(micro_df['NonCommercial_Net'], label='Micro Gold Non-Commercial Net', marker='o', color='orange')
plt.plot(micro_df['Commercial_Net'], label='Micro Gold Commercial Net', marker='x', color='green')
plt.title('Micro Gold Futures COT Net Positions')
plt.xlabel('Entry (All Rows)')
plt.ylabel('Contracts')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# 1️⃣ Paste your COT report here
# -----------------------------
cot_text = """

GOLD - COMMODITY EXCHANGE INC.                                                                                                                   Code-088691
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 100 TROY OUNCES)                                                                                          :
     :          :    Positions                                                                                                           :
All  :   564,585:    21,252     47,000     17,640    242,776     65,403    151,491      9,623     36,914     96,479     23,491    117,723:   57,683    21,656
Old  :   564,585:    21,252     47,000     17,640    242,776     65,403    151,491      9,623     36,914     96,479     23,491    117,723:   57,683    21,656
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :    76,273:     1,487     10,832     -2,241     18,954      3,851      9,671        201      7,303     16,918      2,180     31,241:    8,044     1,712
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       3.8        8.3        3.1       43.0       11.6       26.8        1.7        6.5       17.1        4.2       20.9:     10.2       3.8
Old  :     100.0:       3.8        8.3        3.1       43.0       11.6       26.8        1.7        6.5       17.1        4.2       20.9:     10.2       3.8
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :       317:        22         20         12         30         35         97         17         68         97         38         67:
Old  :       317:        22         20         12         30         35         97         17         68         97         38         67:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 18.6       29.4       28.5       45.6       13.9       26.4       18.8       37.8
Old  :                 18.6       29.4       28.5       45.6       13.9       26.4       18.8       37.8
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
 
MICRO GOLD - COMMODITY EXCHANGE INC.                                                                                                             Code-088695
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 10 TROY OUNCES)                                                                                           :
     :          :    Positions                                                                                                           :
All  :    61,419:       242          0      6,755          0          0        577        202         64     13,601     28,913      5,825:   34,355    26,415
Old  :    61,419:       242          0      6,755          0          0        577        202         64     13,601     28,913      5,825:   34,355    26,415
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :     5,908:         0          0        -55          0          0       -554        202        -30      3,221      2,808         70:    3,256     2,858
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       0.4        0.0       11.0        0.0        0.0        0.9        0.3        0.1       22.1       47.1        9.5:     55.9      43.0
Old  :     100.0:       0.4        0.0       11.0        0.0        0.0        0.9        0.3        0.1       22.1       47.1        9.5:     55.9      43.0
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :        37:         .          0          .          0          0          .          .          .         16         13         13:
Old  :        37:         .          0          .          0          0          .          .          .         16         13         13:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 22.7       44.5       30.9       52.0       21.0       42.6       26.9       45.7
Old  :                 22.7       44.5       30.9       52.0       21.0       42.6       26.9       45.7
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
"""
# -----------------------------
# 2️⃣ Function to parse each section (Gold / Micro Gold)
# -----------------------------
def parse_cot_section(text, section_name):
    section_match = re.search(rf"{section_name}.*?(?=MICRO GOLD|$)", text, re.DOTALL)
    if not section_match:
        print(f"⚠️ Section {section_name} not found!")
        return None
    
    section = section_match.group(0)
    
    # Grab "All" rows (total net positions)
    pattern = r"All\s*:\s*([\d,]+):\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)"
    matches = re.findall(pattern, section)
    
    if not matches:
        print(f"⚠️ No 'All' data found in {section_name}")
        return None
    
    data = []
    for m in matches:
        data.append([int(x.replace(',', '')) for x in m])
    
    columns = [
        "Open_Interest",
        "NonCommercial_Long",
        "NonCommercial_Short",
        "NonCommercial_Spreading",
        "Commercial_Long",
        "Commercial_Short",
        "TotalReportable_Long",
        "TotalReportable_Short"
    ]
    
    df = pd.DataFrame(data, columns=columns)
    
    # Calculate Net Positions
    df['NonCommercial_Net'] = df['NonCommercial_Long'] - df['NonCommercial_Short']
    df['Commercial_Net'] = df['Commercial_Long'] - df['Commercial_Short']
    
    # Extreme thresholds for analysis
    df['Extreme_Long_Threshold'] = df['NonCommercial_Net'].quantile(0.1)
    df['Extreme_Short_Threshold'] = df['NonCommercial_Net'].quantile(0.9)
    
    # Generate signals
    df['Signal'] = 0
    df.loc[df['NonCommercial_Net'] < df['Extreme_Long_Threshold'], 'Signal'] = 1  # buy signal
    df.loc[df['NonCommercial_Net'] > df['Extreme_Short_Threshold'], 'Signal'] = -1  # sell signal
    
    return df

# -----------------------------
# 3️⃣ Parse Gold & Micro Gold
# -----------------------------
gold_df = parse_cot_section(cot_text, "GOLD - COMMODITY EXCHANGE INC")
micro_df = parse_cot_section(cot_text, "MICRO GOLD - COMMODITY EXCHANGE INC")

# -----------------------------
# 4️⃣ Analyze Trends and Predictions
# -----------------------------
def analyze_cot(df, name="Gold"):
    print(f"\n📊 {name} COT Analysis Summary:")
    
    latest = df.iloc[-1]
    print(f"Latest Non-Commercial Net: {latest['NonCommercial_Net']}")
    print(f"Latest Commercial Net: {latest['Commercial_Net']}")
    
    if latest['Signal'] == 1:
        print("✅ Non-Commercials are extremely long → Market may be due for a short-term reversal down")
    elif latest['Signal'] == -1:
        print("✅ Non-Commercials are extremely short → Market may be due for a short-term reversal up")
    else:
        print("⚠️ No extreme positioning → trend likely neutral or continuation")
    
    # Predict next week bias based on trend
    df['NC_Change'] = df['NonCommercial_Net'].diff()
    trend = df['NC_Change'].iloc[-3:].mean()
    
    if trend > 0:
        print("📈 Short-term trend: increasing non-commercial longs → bias slightly bearish")
    elif trend < 0:
        print("📉 Short-term trend: decreasing non-commercial longs → bias slightly bullish")
    else:
        print("➡️ Trend neutral → no clear bias")
    
    # Highlight risky entries: when extremes are reached
    risk_dates = df.index[(df['NonCommercial_Net'] < df['Extreme_Long_Threshold']) |
                          (df['NonCommercial_Net'] > df['Extreme_Short_Threshold'])]
    if len(risk_dates) > 0:
        print("⚠️ Avoid aggressive trading on dates with extreme positioning (high risk):")
        print(risk_dates.tolist())
    
    return df

gold_df = analyze_cot(gold_df, "Gold")
micro_df = analyze_cot(micro_df, "Micro Gold")

# -----------------------------
# 5️⃣ Visualization
# -----------------------------
plt.figure(figsize=(12,5))
plt.plot(gold_df['NonCommercial_Net'], label='Gold Non-Commercial Net', marker='o')
plt.plot(gold_df['Commercial_Net'], label='Gold Commercial Net', marker='x')
plt.axhline(gold_df['Extreme_Long_Threshold'].iloc[0], color='green', linestyle='--', label='Extreme Long Threshold')
plt.axhline(gold_df['Extreme_Short_Threshold'].iloc[0], color='red', linestyle='--', label='Extreme Short Threshold')
plt.title('Gold Futures COT Net Positions')
plt.xlabel('Report Entries')
plt.ylabel('Contracts')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(micro_df['NonCommercial_Net'], label='Micro Gold Non-Commercial Net', marker='o', color='orange')
plt.plot(micro_df['Commercial_Net'], label='Micro Gold Commercial Net', marker='x', color='green')
plt.axhline(micro_df['Extreme_Long_Threshold'].iloc[0], color='green', linestyle='--', label='Extreme Long Threshold')
plt.axhline(micro_df['Extreme_Short_Threshold'].iloc[0], color='red', linestyle='--', label='Extreme Short Threshold')
plt.title('Micro Gold Futures COT Net Positions')
plt.xlabel('Report Entries')
plt.ylabel('Contracts')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (14,6)

# -----------------------------
# 1️⃣ Paste your full COT report here
# -----------------------------
cot_text = """
GOLD - COMMODITY EXCHANGE INC.                                                                                                                   Code-088691
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 100 TROY OUNCES)                                                                                          :
     :          :    Positions                                                                                                           :
All  :   564,585:    21,252     47,000     17,640    242,776     65,403    151,491      9,623     36,914     96,479     23,491    117,723:   57,683    21,656
Old  :   564,585:    21,252     47,000     17,640    242,776     65,403    151,491      9,623     36,914     96,479     23,491    117,723:   57,683    21,656
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :    76,273:     1,487     10,832     -2,241     18,954      3,851      9,671        201      7,303     16,918      2,180     31,241:    8,044     1,712
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       3.8        8.3        3.1       43.0       11.6       26.8        1.7        6.5       17.1        4.2       20.9:     10.2       3.8
Old  :     100.0:       3.8        8.3        3.1       43.0       11.6       26.8        1.7        6.5       17.1        4.2       20.9:     10.2       3.8
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :       317:        22         20         12         30         35         97         17         68         97         38         67:
Old  :       317:        22         20         12         30         35         97         17         68         97         38         67:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 18.6       29.4       28.5       45.6       13.9       26.4       18.8       37.8
Old  :                 18.6       29.4       28.5       45.6       13.9       26.4       18.8       37.8
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
 
MICRO GOLD - COMMODITY EXCHANGE INC.                                                                                                             Code-088695
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 10 TROY OUNCES)                                                                                           :
     :          :    Positions                                                                                                           :
All  :    61,419:       242          0      6,755          0          0        577        202         64     13,601     28,913      5,825:   34,355    26,415
Old  :    61,419:       242          0      6,755          0          0        577        202         64     13,601     28,913      5,825:   34,355    26,415
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :     5,908:         0          0        -55          0          0       -554        202        -30      3,221      2,808         70:    3,256     2,858
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       0.4        0.0       11.0        0.0        0.0        0.9        0.3        0.1       22.1       47.1        9.5:     55.9      43.0
Old  :     100.0:       0.4        0.0       11.0        0.0        0.0        0.9        0.3        0.1       22.1       47.1        9.5:     55.9      43.0
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :        37:         .          0          .          0          0          .          .          .         16         13         13:
Old  :        37:         .          0          .          0          0          .          .          .         16         13         13:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 22.7       44.5       30.9       52.0       21.0       42.6       26.9       45.7
Old  :                 22.7       44.5       30.9       52.0       21.0       42.6       26.9       45.7
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
"""

# -----------------------------
# 2️⃣ Parse COT Section Function
# -----------------------------
def parse_cot_section(text, section_name):
    # Grab section
    section_match = re.search(rf"{section_name}.*?(?=MICRO GOLD|$)", text, re.DOTALL)
    if not section_match:
        print(f"⚠️ Section {section_name} not found!")
        return None
    
    section = section_match.group(0)
    
    # Extract the "All" row (total positions)
    pattern = r"All\s*:\s*([\d,]+):\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)"
    matches = re.findall(pattern, section)
    
    if not matches:
        print(f"⚠️ No 'All' data found in {section_name}")
        return None
    
    data = []
    for i, m in enumerate(matches):
        numbers = [int(x.replace(',', '')) for x in m]
        # Use sequential indices as pseudo-dates (we'll replace later)
        data.append([f"Week_{i+1}"] + numbers)
    
    columns = ["Week","Open_Interest","NonCommercial_Long","NonCommercial_Short","NonCommercial_Spreading",
               "Commercial_Long","Commercial_Short","TotalReportable_Long","TotalReportable_Short"]
    
    df = pd.DataFrame(data, columns=columns)
    
    # Calculate net positions
    df['NonCommercial_Net'] = df['NonCommercial_Long'] - df['NonCommercial_Short']
    df['Commercial_Net'] = df['Commercial_Long'] - df['Commercial_Short']
    
    # Compute extremes
    df['Extreme_Long_Threshold'] = df['NonCommercial_Net'].quantile(0.1)
    df['Extreme_Short_Threshold'] = df['NonCommercial_Net'].quantile(0.9)
    
    # Signal: 1=buy, -1=sell, 0=neutral
    df['Signal'] = 0
    df.loc[df['NonCommercial_Net'] < df['Extreme_Long_Threshold'], 'Signal'] = 1
    df.loc[df['NonCommercial_Net'] > df['Extreme_Short_Threshold'], 'Signal'] = -1
    
    # Probability strength (scaled from extreme percentile)
    df['Long_Prob'] = np.clip((df['Extreme_Short_Threshold'] - df['NonCommercial_Net']) / 
                              (df['Extreme_Short_Threshold'] - df['Extreme_Long_Threshold']),0,1)
    df['Short_Prob'] = np.clip((df['NonCommercial_Net'] - df['Extreme_Long_Threshold']) / 
                               (df['Extreme_Short_Threshold'] - df['Extreme_Long_Threshold']),0,1)
    
    return df

# -----------------------------
# 3️⃣ Parse Gold & Micro Gold
# -----------------------------
gold_df = parse_cot_section(cot_text, "GOLD - COMMODITY EXCHANGE INC")
micro_df = parse_cot_section(cot_text, "MICRO GOLD - COMMODITY EXCHANGE INC")

# -----------------------------
# 4️⃣ Analysis Function
# -----------------------------
def analyze_cot(df, name="Gold"):
    latest = df.iloc[-1]
    print(f"\n📊 {name} COT Professional Analysis:")
    
    print(f"Latest Week: {latest['Week']}")
    print(f"Non-Commercial Net Position: {latest['NonCommercial_Net']} contracts")
    print(f"Commercial Net Position: {latest['Commercial_Net']} contracts")
    
    # Signals
    if latest['Signal'] == 1:
        print("✅ Extreme Non-Commercial Longs → Market may reverse down soon (short-term caution).")
    elif latest['Signal'] == -1:
        print("✅ Extreme Non-Commercial Shorts → Market may reverse up soon (short-term opportunity).")
    else:
        print("➡️ No extreme positioning → trend likely neutral or continuing.")
    
    # Probability-based bias
    print(f"📈 Suggested Bias Strengths: Long: {latest['Long_Prob']*100:.1f}% | Short: {latest['Short_Prob']*100:.1f}%")
    
    # Identify risky weeks
    risky_weeks = df['Week'][(df['NonCommercial_Net'] < df['Extreme_Long_Threshold']) |
                             (df['NonCommercial_Net'] > df['Extreme_Short_Threshold'])].tolist()
    if risky_weeks:
        print("⚠️ Weeks with extreme positioning to avoid aggressive trading:")
        print(risky_weeks)
    
    # Trend explanation
    df['NC_Change'] = df['NonCommercial_Net'].diff()
    trend = df['NC_Change'].iloc[-3:].mean()
    if trend > 0:
        print("📉 Short-term Trend: increasing Non-Commercial longs → slightly bearish continuation risk.")
    elif trend < 0:
        print("📈 Short-term Trend: decreasing Non-Commercial longs → slightly bullish continuation possible.")
    else:
        print("➡️ Short-term Trend neutral → market likely consolidating.")
    
    return df

gold_df = analyze_cot(gold_df, "Gold")
micro_df = analyze_cot(micro_df, "Micro Gold")

# -----------------------------
# 5️⃣ Professional-Style Visualization
# -----------------------------
def plot_cot(df, name="Gold"):
    plt.figure(figsize=(14,6))
    sns.lineplot(x='Week', y='NonCommercial_Net', data=df, marker='o', label='Non-Commercial Net', linewidth=2)
    sns.lineplot(x='Week', y='Commercial_Net', data=df, marker='x', label='Commercial Net', linewidth=2)
    
    plt.axhline(df['Extreme_Long_Threshold'].iloc[0], color='green', linestyle='--', label='Extreme Long Threshold')
    plt.axhline(df['Extreme_Short_Threshold'].iloc[0], color='red', linestyle='--', label='Extreme Short Threshold')
    
    # Annotate risky weeks
    for idx,row in df.iterrows():
        if row['Signal'] != 0:
            plt.text(idx, row['NonCommercial_Net'], f"{row['Week']}", color='black', fontsize=10)
    
    plt.title(f"{name} COT Net Positions and Extreme Thresholds", fontsize=16)
    plt.xlabel("Weeks")
    plt.ylabel("Contracts")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_cot(gold_df, "Gold")
plot_cot(micro_df, "Micro Gold")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1️⃣ Historical Gold Data
# -----------------------------
# Example: daily gold closes
# Replace with your actual gold historical price series
gold_prices = pd.Series([5200, 5210, 5225, 5215, 5230, 5240, 5235, 5250, 5265, 5270])

# Calculate daily returns
returns = gold_prices.pct_change().dropna()
mu = returns.mean()
sigma = returns.std()
print(f"Mean daily return: {mu:.4f}, Volatility: {sigma:.4f}")

# -----------------------------
# 2️⃣ Monte Carlo Simulation
# -----------------------------
np.random.seed(42)
S0 = gold_prices.iloc[-1]  # last price
T = 20  # forecast 20 days
simulations = 5000

simulated_paths = np.zeros((T, simulations))
for i in range(simulations):
    path = [S0]
    for t in range(1, T):
        shock = np.random.normal(mu, sigma)
        price = path[-1] * (1 + shock)
        path.append(price)
    simulated_paths[:, i] = path

# -----------------------------
# 3️⃣ Visualization
# -----------------------------
plt.figure(figsize=(14,6))
plt.plot(simulated_paths, color='lightgray', alpha=0.1)
plt.plot(simulated_paths.mean(axis=1), color='red', linewidth=2, label='Average Path')
plt.title("Monte Carlo Simulations for Gold (Next 20 Days)", fontsize=16)
plt.xlabel("Days")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

# -----------------------------
# 4️⃣ Probability of Going Long / Short
# -----------------------------
prob_up = np.mean(simulated_paths[-1, :] > S0) * 100
prob_down = np.mean(simulated_paths[-1, :] < S0) * 100
print(f"📊 Monte Carlo Prediction: Probability Gold ↑: {prob_up:.1f}% | Probability Gold ↓: {prob_down:.1f}%")


In [ ]:
# -----------------------------
# 1️⃣ Paste your Oil COT report here
# -----------------------------
oil_text = """
WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE                                                                                           Code-06765A
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 1,000 BARRELS)                                                                                            :
     :          :    Positions                                                                                                           :
All  :   193,947:    36,060     48,475     61,843     52,716     10,393          0          0          0     45,755     22,529     14,597:   25,299    45,237
Old  :   193,947:    36,060     48,475     61,843     52,716     10,393          0          0          0     45,755     22,529     14,597:   25,299    45,237
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :     4,334:     2,171      2,370        279        468        -46          0          0          0        704        511      1,129:       97       -98
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:      18.6       25.0       31.9       27.2        5.4        0.0        0.0        0.0       23.6       11.6        7.5:     13.0      23.3
Old  :     100.0:      18.6       25.0       31.9       27.2        5.4        0.0        0.0        0.0       23.6       11.6        7.5:     13.0      23.3
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :        44:        14         14         12          4         10          0          0          0          4          6          7:
Old  :        44:        14         14         12          4         10          0          0          0          4          6          7:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 32.5       35.7       49.0       50.1       28.9       35.7       43.5       49.9
Old  :                 32.5       35.7       49.0       50.1       28.9       35.7       43.5       49.9
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
 
CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE                                                                                       Code-06765C
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 1,000 BARRELS)                                                                                            :
     :          :    Positions                                                                                                           :
All  :   153,113:     8,392     19,875     38,734     77,194     20,372     10,128          0      1,125     43,732          0     24,170:    6,460    10,377
Old  :   153,113:     8,392     19,875     38,734     77,194     20,372     10,128          0      1,125     43,732          0     24,170:    6,460    10,377
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :    10,399:       237      5,589      4,508      8,077     -2,546        512          0        135      8,452        -78     -1,348:      448       569
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       5.5       13.0       25.3       50.4       13.3        6.6        0.0        0.7       28.6        0.0       15.8:      4.2       6.8
Old  :     100.0:       5.5       13.0       25.3       50.4       13.3        6.6        0.0        0.7       28.6        0.0       15.8:      4.2       6.8
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :        32:         .          5          4         12         16          .          0          .          8          0          7:
Old  :        32:         .          5          4         12         16          .          0          .          8          0          7:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 41.2       32.5       66.3       54.4       35.5       30.8       53.7       51.6
Old  :                 41.2       32.5       66.3       54.4       35.5       30.8       53.7       51.6
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
 
BRENT LAST DAY - NEW YORK MERCANTILE EXCHANGE                                                                                                    Code-06765T
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 1,000 BARRELS)                                                                                            :
     :          :    Positions                                                                                                           :
All  :   302,819:    27,406     62,881     51,632      5,892     40,583      8,732      1,102      6,210     36,820     56,835    120,014:   11,421     9,302
Old  :   302,819:    27,406     62,881     51,632      5,892     40,583      8,732      1,102      6,210     36,820     56,835    120,014:   11,421     9,302
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :    20,418:     3,697      7,685      7,110      4,710     -1,827       -359         41     -1,307      3,518        614      8,339:    1,246     2,163
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:       9.1       20.8       17.1        1.9       13.4        2.9        0.4        2.1       12.2       18.8       39.6:      3.8       3.1
Old  :     100.0:       9.1       20.8       17.1        1.9       13.4        2.9        0.4        2.1       12.2       18.8       39.6:      3.8       3.1
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :        56:        11         12          9          .          9          .          .          5         19          9         26:
Old  :        56:        11         12          9          .          9          .          .          5         19          9         26:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 33.7       43.8       52.8       61.0       17.0       26.4       25.8       32.3
Old  :                 33.7       43.8       52.8       61.0       17.0       26.4       25.8       32.3
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
 """

# -----------------------------
# 2️⃣ Flexible Oil COT parser
# -----------------------------
def parse_oil_cot(text, section_keywords=["CRUDE OIL", "WTI", "BRENT"]):
    # Search for first section containing any keyword
    section_match = None
    for kw in section_keywords:
        section_match = re.search(rf"{kw}.*?(?=(CRUDE OIL AVG|WTI-BRENT|$))", text, re.DOTALL | re.IGNORECASE)
        if section_match:
            break
    if not section_match:
        print("⚠️ Oil section not found!")
        return None
    
    section = section_match.group(0)
    
    # Extract the "All" row (total positions)
    pattern = r"All\s*:\s*([\d,]+):\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)\s*([\d,]+)"
    matches = re.findall(pattern, section)
    
    if not matches:
        print("⚠️ No 'All' data found in Oil section")
        return None
    
    data = []
    for i, m in enumerate(matches):
        numbers = [int(x.replace(',', '')) for x in m]
        data.append([f"Week_{i+1}"] + numbers)
    
    # Define columns (simplified to main categories for analysis)
    columns = ["Week","Open_Interest",
               "Prod_Long","Prod_Short","Prod_Spread",
               "Swap_Long","Swap_Short","Swap_Spread",
               "MM_Long","MM_Short","MM_Spread",
               "Other_Long","Other_Short"]
    
    df = pd.DataFrame(data, columns=columns)
    
    # Net positions for Managed Money (the main speculator group)
    df['MM_Net'] = df['MM_Long'] - df['MM_Short']
    
    # Signal thresholds
    df['Extreme_Long_Threshold'] = df['MM_Net'].quantile(0.1)
    df['Extreme_Short_Threshold'] = df['MM_Net'].quantile(0.9)
    
    df['Signal'] = 0
    df.loc[df['MM_Net'] < df['Extreme_Long_Threshold'], 'Signal'] = 1
    df.loc[df['MM_Net'] > df['Extreme_Short_Threshold'], 'Signal'] = -1
    
    df['Long_Prob'] = np.clip((df['Extreme_Short_Threshold'] - df['MM_Net']) / 
                              (df['Extreme_Short_Threshold'] - df['Extreme_Long_Threshold']),0,1)
    df['Short_Prob'] = np.clip((df['MM_Net'] - df['Extreme_Long_Threshold']) / 
                               (df['Extreme_Short_Threshold'] - df['Extreme_Long_Threshold']),0,1)
    
    return df

# -----------------------------
# 3️⃣ Parse Oil COT
# -----------------------------
oil_df = parse_oil_cot(oil_text)

# -----------------------------
# 4️⃣ Analyze Oil COT
# -----------------------------
def analyze_oil_cot(df, name="Oil"):
    if df is None:
        print(f"⚠️ No data for {name}")
        return None
    
    latest = df.iloc[-1]
    print(f"\n📊 {name} COT Professional Analysis:")
    print(f"Latest Week: {latest['Week']}")
    print(f"Managed Money Net Position: {latest['MM_Net']} contracts")
    
    if latest['Signal'] == 1:
        print("✅ Extreme Managed Money Longs → potential short-term pullback.")
    elif latest['Signal'] == -1:
        print("✅ Extreme Managed Money Shorts → potential short-term rally.")
    else:
        print("➡️ No extreme positioning → trend likely neutral or continuing.")
    
    print(f"📈 Suggested Bias Strengths: Long: {latest['Long_Prob']*100:.1f}% | Short: {latest['Short_Prob']*100:.1f}%")
    
    # Identify risky weeks
    risky_weeks = df['Week'][(df['MM_Net'] < df['Extreme_Long_Threshold']) |
                             (df['MM_Net'] > df['Extreme_Short_Threshold'])].tolist()
    if risky_weeks:
        print("⚠️ Weeks with extreme MM positioning to avoid aggressive trading:")
        print(risky_weeks)
    
    # Short-term trend
    df['MM_Change'] = df['MM_Net'].diff()
    trend = df['MM_Change'].iloc[-3:].mean()
    if trend > 0:
        print("📉 Short-term Trend: increasing MM longs → slightly bearish continuation risk.")
    elif trend < 0:
        print("📈 Short-term Trend: decreasing MM longs → slightly bullish continuation possible.")
    else:
        print("➡️ Short-term Trend neutral → market likely consolidating.")
    
    return df

oil_df = analyze_oil_cot(oil_df, "Oil")


In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1️⃣ Paste your COT report here
# -----------------------------
cot_report = """

WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE                                                                                           Code-06765A
Disaggregated Commitments of Traders - Options and Futures Combined, August 11, 2026                                                                         
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :                                              Reportable Positions                                                      :   Nonreportable
     :          :  Producer/Merchant/ :                                :                                :                                :     Positions
     :   Open   :   Processor/User    :          Swap Dealers          :         Managed Money          :       Other Reportables        :
     : Interest :   Long   :  Short   :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long   :  Short   :Spreading :   Long  :  Short
-------------------------------------------------------------------------------------------------------------------------------------------------------------
     :          :(CONTRACTS OF 1,000 BARRELS)                                                                                            :
     :          :    Positions                                                                                                           :
All  :   193,947:    36,060     48,475     61,843     52,716     10,393          0          0          0     45,755     22,529     14,597:   25,299    45,237
Old  :   193,947:    36,060     48,475     61,843     52,716     10,393          0          0          0     45,755     22,529     14,597:   25,299    45,237
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:        0         0
     :          :                                                                                                                        :
     :          :    Changes in Commitments from:       August 4, 2026                                                                   :
     :     4,334:     2,171      2,370        279        468        -46          0          0          0        704        511      1,129:       97       -98
     :          :                                                                                                                        :
     :          :    Percent of Open Interest Represented by Each Category of Trader                                                     :
All  :     100.0:      18.6       25.0       31.9       27.2        5.4        0.0        0.0        0.0       23.6       11.6        7.5:     13.0      23.3
Old  :     100.0:      18.6       25.0       31.9       27.2        5.4        0.0        0.0        0.0       23.6       11.6        7.5:     13.0      23.3
Other:     100.0:       0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0:      0.0       0.0
     :          :                                                                                                                        :
     :          :    Number of Traders in Each Category                                                                                  :
All  :        44:        14         14         12          4         10          0          0          0          4          6          7:
Old  :        44:        14         14         12          4         10          0          0          0          4          6          7:
Other:         0:         0          0          0          0          0          0          0          0          0          0          0:
     :-------------------------------------------------------------------------------------------------------------------------------------------------------
     :             Percent of Open Interest Held by the Indicated Number of the Largest Traders
     :                          By Gross Position                       By Net Position
     :               4 or Less Traders     8 or Less Traders     4 or Less Traders     8 or Less Traders
     :                 Long:     Short       Long      Short:      Long      Short       Long      Short
     :----------------------------------------------------------------------------------------------------
All  :                 32.5       35.7       49.0       50.1       28.9       35.7       43.5       49.9
Old  :                 32.5       35.7       49.0       50.1       28.9       35.7       43.5       49.9
Other:                  0.0        0.0        0.0        0.0        0.0        0.0        0.0        0.0
"""

# -----------------------------
# 2️⃣ Parse the COT report
# -----------------------------
def parse_cot(report, section_name="WTI FINANCIAL CRUDE OIL"):
    # Find section by name (case insensitive)
    section_pattern = re.compile(section_name, re.IGNORECASE)
    match = section_pattern.search(report)
    if not match:
        raise ValueError(f"Section '{section_name}' not found!")
    
    section_start = match.end()
    section_text = report[section_start:]
    
    # Extract "All" positions line (any numbers)
    all_match = re.search(r"All\s*:\s*([\d,]+)\s*:(.*)", section_text)
    if not all_match:
        raise ValueError("Could not parse 'All' line in section")
    
    all_numbers = all_match.group(2)
    # Extract all numeric values (ignore colons/spaces)
    numbers = [int(x.replace(",", "")) for x in re.findall(r"[\d,]+", all_numbers)]
    
    # Dynamically create category names
    categories = [
        "Producer/ Merchant Long", "Producer/ Merchant Short",
        "Processor/ User Long", "Processor/ User Short",
        "Swap Dealers Long", "Swap Dealers Short", "Swap Dealers Spreading",
        "Managed Money Long", "Managed Money Short", "Managed Money Spreading",
        "Other Reportables Long", "Other Reportables Short", "Other Reportables Spreading",
        "Nonreportable Long", "Nonreportable Short"
    ]
    
    # Trim or pad categories to match number of values
    if len(numbers) < len(categories):
        categories = categories[:len(numbers)]
    elif len(numbers) > len(categories):
        categories += [f"Extra_{i}" for i in range(len(numbers) - len(categories))]
    
    df = pd.DataFrame({
        "Category": categories,
        "Contracts": numbers
    })
    
    return df

# -----------------------------
# 3️⃣ Monte Carlo simulation
# -----------------------------
def monte_carlo_sim(df, n_sim=1000, n_weeks=12):
    last_value = df["Contracts"].sum()
    returns = np.diff(df["Contracts"].values) / df["Contracts"].values[:-1] if len(df) > 1 else np.array([0])
    
    if len(returns) == 0:
        returns = np.array([0])
    
    sim_matrix = np.zeros((n_weeks, n_sim))
    
    for i in range(n_sim):
        values = [last_value]
        for _ in range(n_weeks):
            shock = np.random.choice(returns)
            values.append(values[-1] * (1 + shock))
        sim_matrix[:, i] = values[1:]
    
    return sim_matrix

# -----------------------------
# 4️⃣ Plotting
# -----------------------------
def plot_cot(df, sim_matrix):
    plt.figure(figsize=(12,6))
    plt.bar(df["Category"], df["Contracts"], color='gold', alpha=0.7)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel("Contracts (1,000 barrels)")
    plt.title("WTI COT Positions")
    plt.tight_layout()
    plt.show()
    
    # Monte Carlo simulations
    plt.figure(figsize=(12,6))
    for i in range(min(sim_matrix.shape[1], 100)):
        plt.plot(sim_matrix[:,i], color='blue', alpha=0.05)
    plt.title("Monte Carlo Simulations of Total Open Interest")
    plt.ylabel("Contracts (1,000 barrels)")
    plt.xlabel("Weeks Ahead")
    plt.show()

# -----------------------------
# 5️⃣ Run full analysis
# -----------------------------
oil_df = parse_cot(cot_report)
oil_sim = monte_carlo_sim(oil_df)
plot_cot(oil_df, oil_sim)


In [ ]:
# -----------------------------
# 0️⃣ Install required libraries if not already installed
# -----------------------------
# !pip install yfinance pandas numpy matplotlib seaborn

# -----------------------------
# 1️⃣ Imports
# -----------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns

sns.set(style="whitegrid")

# -----------------------------
# 2️⃣ Functions
# -----------------------------

# Fetch Gold Futures Price
def get_gold_data(period="3mo", interval="1d"):
    gold = yf.Ticker("GC=F")  # Gold Futures continuous contract
    df = gold.history(period=period, interval=interval)
    df = df[["Close"]]
    df.rename(columns={"Close":"Gold_Price"}, inplace=True)
    df["Gold_Return"] = df["Gold_Price"].pct_change()
    return df.dropna()

# ----- Fetch 10Y US Treasury Yield -----
def get_10y_yield(period="3mo"):
    yield10 = yf.Ticker("^TNX")
    df = yield10.history(period=period)
    df = df[["Close"]]
    df.rename(columns={"Close":"Yield_10Y"}, inplace=True)
    df["Yield_Change"] = df["Yield_10Y"].pct_change()
    return df.dropna()

# ----- Fetch USD Index -----
def get_usd_index(period="3mo"):
    dxy = yf.Ticker("DX-Y.NYB")
    df = dxy.history(period=period)
    df = df[["Close"]]
    df.rename(columns={"Close":"USD_Index"}, inplace=True)
    df["USD_Return"] = df["USD_Index"].pct_change()
    return df.dropna()

# ----- Fetch WTI Crude (Oil) -----
def get_oil_data(period="3mo"):
    oil = yf.Ticker("CL=F")
    df = oil.history(period=period)
    df = df[["Close"]]
    df.rename(columns={"Close":"WTI_Price"}, inplace=True)
    df["WTI_Return"] = df["WTI_Price"].pct_change()
    return df.dropna()

# ----- Merge All Data -----
def merge_data(gold, yield10, usd, oil):
    df = gold.join(yield10, how="outer")
    df = df.join(usd, how="outer")
    df = df.join(oil, how="outer")
    df = df.fillna(method="ffill").dropna()
    return df

# ----- Monte Carlo Forecast for Gold -----
def monte_carlo_gold(df, n_sim=500, n_days=10):
    last_price = df["Gold_Price"].iloc[-1]
    returns = df["Gold_Return"].values
    if len(returns) == 0:
        returns = np.array([0])
    
    sim_matrix = np.zeros((n_days, n_sim))
    for i in range(n_sim):
        prices = [last_price]
        for _ in range(n_days):
            shock = np.random.choice(returns)
            prices.append(prices[-1]*(1+shock))
        sim_matrix[:, i] = prices[1:]
    return sim_matrix

# ----- Plot Forecast -----
def plot_forecast(df, sim_matrix):
    plt.figure(figsize=(12,6))
    # Monte Carlo simulations
    for i in range(min(sim_matrix.shape[1], 200)):
        plt.plot(sim_matrix[:,i], color='blue', alpha=0.05)
    plt.title("Monte Carlo Forecast of Gold Price (Next 20 Days)")
    plt.ylabel("Gold Price (USD)")
    plt.xlabel("Days Ahead")
    plt.show()
    
    # Correlation heatmap
    plt.figure(figsize=(8,6))
    corr = df[["Gold_Price", "Yield_10Y", "USD_Index", "WTI_Price"]].corr()
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Analysis")
    plt.show()

# -----------------------------
# 3️⃣ Run Analysis
# -----------------------------
gold_df = get_gold_data()
yield_df = get_10y_yield()
usd_df = get_usd_index()
oil_df = get_oil_data()

full_df = merge_data(gold_df, yield_df, usd_df, oil_df)
print("Last 5 rows of merged data:")
display(full_df.tail())

sim_matrix = monte_carlo_gold(full_df)
plot_forecast(full_df, sim_matrix)


In [ ]:
!pip install statsmodels
!pip install pandas matplotlib numpy


In [ ]:
!pip install arch


In [ ]:
# -----------------------------
# 1️⃣ Imports
# -----------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
import statsmodels.api as sm

# -----------------------------
# 2️⃣ Prepare your COT-based gold data
# -----------------------------
# Example: replace with your parsed COT 'All' positions over time
# For demonstration, we'll make a small DataFrame
# Replace these dates and positions with your actual parsed COT weekly data

# WAS: 6 hardcoded placeholder prices left over from a demo. Every fit below
# was running on them, which is where "Probability Gold up: 100.0%" came from.
_raw = yf.download("GC=F", period="5y", interval="1wk", progress=False)
_close = _raw["Close"]
if isinstance(_close, pd.DataFrame):
    _close = _close.iloc[:, 0]

df = pd.DataFrame({'Gold_Price': _close.dropna()})
df.index.name = 'Date'
need_obs(len(df), 250, "FUNDAMENTALS weekly gold")

# -----------------------------
# 3️⃣ Compute returns
# -----------------------------
df['Gold_Return'] = df['Gold_Price'].pct_change().fillna(0)

# -----------------------------
# 4️⃣ GARCH(1,1) model for gold volatility
# -----------------------------
garch_model = arch_model(df['Gold_Return']*100, vol='Garch', p=1, q=1)
garch_res = garch_model.fit(disp='off')
need_obs(len(df), 250, 'GARCH(1,1)')
df['Volatility'] = garch_res.conditional_volatility

# Forecast 10-day volatility
garch_forecast = garch_res.forecast(horizon=10)
vol_forecast = garch_forecast.variance.iloc[-1].values
print("10-day forecasted gold volatility (%):", vol_forecast)

# -----------------------------
# 5️⃣ Regime-switching model
# -----------------------------
reg_model = MarkovRegression(df['Gold_Return'], k_regimes=2, trend='c', switching_variance=True)
reg_res = reg_model.fit(disp=False)
need_converged(reg_res, label='MarkovRegression')
df['Regime'] = reg_res.smoothed_marginal_probabilities[1]  # probability of being in regime 1

# -----------------------------
# 6️⃣ Macro factor model fallback (self-regression)
# -----------------------------
# Since no other factors, we can do a simple autoregressive linear regression
X = sm.add_constant(df['Gold_Return'].shift(1).fillna(0))
y = df['Gold_Return']
need_obs(len(y), 30, 'gold AR(1)')
macro_model = sm.OLS(y, X).fit()
df['Predicted_Return'] = macro_model.predict(X)
print("\nAuto-regression summary:")
print(macro_model.summary())

# -----------------------------
# 7️⃣ Forecast next 10 days
# -----------------------------
last_return = df['Gold_Return'].iloc[-1]
last_price = df['Gold_Price'].iloc[-1]

pred_returns = []
for _ in range(10):
    shock = np.random.normal(0, df['Volatility'].iloc[-1]/100)
    pred = macro_model.predict([1, last_return])[0] + shock
    pred_returns.append(pred)
    last_return = pred  # feed predicted return into next step

pred_prices = [last_price]
for r in pred_returns:
    pred_prices.append(pred_prices[-1]*(1+r))

# -----------------------------
# 8️⃣ Plot historical + forecast
# -----------------------------
plt.figure(figsize=(12,6))
plt.plot(df['Gold_Price'], label='Historical Gold Price', marker='o')
plt.plot(pd.date_range(df.index[-1], periods=11, freq='D'), pred_prices, label='Forecast 10 Days', linestyle='--', marker='x')
plt.title('Gold Price Forecast (10 Days) based on COT & GARCH')
plt.xlabel('Date')
plt.ylabel('Gold Price')
plt.legend()
plt.show()


In [ ]:
import yfinance as yf

# Fetch gold futures hourly data
gold = yf.download("GC=F", start="2026-02-01", end="2026-03-13", interval="1h")
print(gold.head())


In [ ]:
# -----------------------------
# 1️⃣ Imports
# -----------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
import statsmodels.api as sm
import yfinance as yf

# -----------------------------
# 2️⃣ Fetch actual gold futures prices safely
# -----------------------------
gold = yf.download("GC=F", start="2026-02-01", end="2026-03-13", interval="1h")

# Check which column exists
if 'Adj Close' in gold.columns:
    df = gold[['Adj Close']].rename(columns={'Adj Close':'Gold_Price'})
elif 'Close' in gold.columns:
    df = gold[['Close']].rename(columns={'Close':'Gold_Price'})
else:
    raise ValueError("No suitable price column found in the data!")

df['Gold_Return'] = df['Gold_Price'].pct_change().fillna(0)
print(df.head())

# -----------------------------
# 3️⃣ GARCH(1,1) model for volatility
# -----------------------------
garch_model = arch_model(df['Gold_Return']*100, vol='Garch', p=1, q=1)
garch_res = garch_model.fit(disp='off')
df['Volatility'] = garch_res.conditional_volatility

# Forecast 10-day volatility
garch_forecast = garch_res.forecast(horizon=10)
vol_forecast = garch_forecast.variance.iloc[-1].values
print("10-day forecasted gold volatility (%):", vol_forecast)

# -----------------------------
# 4️⃣ Regime-switching model
# -----------------------------
reg_model = MarkovRegression(df['Gold_Return'], k_regimes=2, trend='c', switching_variance=True)
reg_res = reg_model.fit(disp=False)
df['Regime'] = reg_res.smoothed_marginal_probabilities[1]

# -----------------------------
# 5️⃣ Simple autoregressive model (fallback macro)
# -----------------------------
X = sm.add_constant(df['Gold_Return'].shift(1).fillna(0))
y = df['Gold_Return']
macro_model = sm.OLS(y, X).fit()
df['Predicted_Return'] = macro_model.predict(X)
print("\nAuto-regression summary:")
print(macro_model.summary())

# -----------------------------
# 6️⃣ Forecast next 10 days
# -----------------------------
last_return = df['Gold_Return'].iloc[-1]
last_price = df['Gold_Price'].iloc[-1]
pred_returns = []

for _ in range(10):
    shock = np.random.normal(0, df['Volatility'].iloc[-1]/100)
    pred = macro_model.predict([1, last_return])[0] + shock
    pred_returns.append(pred)
    last_return = pred

pred_prices = [last_price]
for r in pred_returns:
    pred_prices.append(pred_prices[-1]*(1+r))

# -----------------------------
# 7️⃣ Plot historical + forecast
# -----------------------------
plt.figure(figsize=(12,6))
plt.plot(df['Gold_Price'], label='Historical Gold Price')
plt.plot(pd.date_range(df.index[-1], periods=11, freq='D'), pred_prices, label='Forecast 10 Days', linestyle='--')
plt.title('Gold Futures Price Forecast (10 Days)')
plt.xlabel('Date')
plt.ylabel('Gold Price ($)')
plt.legend()
plt.show()


In [ ]:
pip install seaborn


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select numeric columns
heatmap_data = df.select_dtypes(include=[float, int])

# Compute correlation matrix
corr = heatmap_data.corr()

# Plot heatmap
plt.figure(figsize=(10,8))
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Gold Model Correlation Heatmap")
plt.show()


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1️⃣ Download gold futures
# -----------------------------
gold = yf.download("GC=F", period="30d", interval="1h")

# Flatten columns if multi-index
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):
    gold.columns = gold.columns.get_level_values(0)

# -----------------------------
# 2️⃣ Extract price and volume
# -----------------------------
price = gold["Close"]
volume = gold["Volume"]

df = pd.DataFrame({
    "price": price.values,
    "volume": volume.values
}, index=price.index)

# -----------------------------
# 3️⃣ Create price bins
# -----------------------------
bins = np.linspace(df.price.min(), df.price.max(), 50)

df["price_bin"] = pd.cut(df["price"], bins)

# -----------------------------
# 4️⃣ Volume aggregation
# -----------------------------
volume_profile = df.groupby("price_bin", observed=True)["volume"].sum()

price_levels = [b.mid for b in volume_profile.index]

heatmap_df = pd.DataFrame({
    "price": price_levels,
    "volume": volume_profile.values
})

# -----------------------------
# 5️⃣ Plot heatmap
# -----------------------------
plt.figure(figsize=(6,10))

sns.heatmap(
    heatmap_df[["volume"]],
    yticklabels=np.round(heatmap_df["price"],2),
    cmap="inferno"
)

plt.title("Gold Futures Liquidity Heatmap")
plt.xlabel("Volume Liquidity")
plt.ylabel("Price Level")

plt.show()


In [ ]:
# ===============================
# 1️⃣ Imports
# ===============================
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from arch import arch_model

# ===============================
# 2️⃣ Download Gold Futures Data
# ===============================
gold = yf.download("GC=F", period="30d", interval="1h")

# Flatten columns if multi-index
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):
    gold.columns = gold.columns.get_level_values(0)

df = gold[['Open','High','Low','Close','Volume']].copy()

# ===============================
# 3️⃣ Volume Profile (Institutional Liquidity)
# ===============================
bins = np.linspace(df.Close.min(), df.Close.max(), 60)

df["price_bin"] = pd.cut(df["Close"], bins)

volume_profile = df.groupby("price_bin", observed=False)["Volume"].sum()

price_levels = [b.mid for b in volume_profile.index]

volume_heatmap = pd.DataFrame({
    "price": price_levels,
    "volume": volume_profile.values
})

# ===============================
# 4️⃣ Swing High / Low Detection
# (Retail Stop Zones)
# ===============================
window = 10

df["Swing_High"] = df["High"][(df["High"] == df["High"].rolling(window, center=True).max())]
df["Swing_Low"] = df["Low"][(df["Low"] == df["Low"].rolling(window, center=True).min())]

# ===============================
# 5️⃣ VWAP (Institutional Execution)
# ===============================
df["Cum_Vol"] = df["Volume"].cumsum()
df["Cum_Vol_Price"] = (df["Close"] * df["Volume"]).cumsum()

df["VWAP"] = df["Cum_Vol_Price"] / df["Cum_Vol"]

df["VWAP_STD"] = df["Close"].rolling(50).std()

df["VWAP_upper"] = df["VWAP"] + 2 * df["VWAP_STD"]
df["VWAP_lower"] = df["VWAP"] - 2 * df["VWAP_STD"]

# ===============================
# 6️⃣ Liquidity Void Detection
# ===============================
df["Range"] = df["High"] - df["Low"]
df["Volume_MA"] = df["Volume"].rolling(20).mean()

df["Liquidity_Void"] = (df["Range"] > df["Range"].rolling(20).mean()*2) & (df["Volume"] < df["Volume_MA"])

# ===============================
# 7️⃣ Volatility Model (GARCH)
# ===============================
returns = df["Close"].pct_change().dropna()*100

garch = arch_model(returns, p=1, q=1)
garch_fit = garch.fit(disp="off")

df["Volatility"] = np.nan
df.loc[df.index[1:], "Volatility"] = garch_fit.conditional_volatility

print("Current estimated volatility:", df["Volatility"].iloc[-1])

# ===============================
# 8️⃣ Plot Liquidity Radar
# ===============================

plt.figure(figsize=(14,7))

# Price
plt.plot(df.index, df["Close"], label="Gold Price", color="black")

# VWAP
plt.plot(df.index, df["VWAP"], label="VWAP", color="blue")

plt.plot(df.index, df["VWAP_upper"], linestyle="--", color="green")
plt.plot(df.index, df["VWAP_lower"], linestyle="--", color="red")

# Swing highs/lows
plt.scatter(df.index, df["Swing_High"], color="red", label="Retail Stops (Highs)")
plt.scatter(df.index, df["Swing_Low"], color="green", label="Retail Stops (Lows)")

# Liquidity voids
voids = df[df["Liquidity_Void"]]
plt.scatter(voids.index, voids["Close"], color="purple", label="Liquidity Void")

plt.title("Gold Liquidity Radar")
plt.legend()
plt.show()

# ===============================
# 9️⃣ Volume Heatmap
# ===============================

plt.figure(figsize=(6,10))

sns.heatmap(
    volume_heatmap[["volume"]],
    yticklabels=np.round(volume_heatmap["price"],2),
    cmap="inferno"
)

plt.title("Gold Volume Liquidity Heatmap")
plt.xlabel("Volume")
plt.ylabel("Price Level")

plt.show()


In [ ]:
# =====================================
# 1️⃣ Imports
# =====================================
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================
# 2️⃣ Download Gold Futures Data
# =====================================
gold = yf.download("GC=F", period="60d", interval="1h")

# Flatten columns if multi-index
if hasattr(gold, 'columns') and isinstance(gold.columns, pd.MultiIndex):
    gold.columns = gold.columns.get_level_values(0)

df = gold[['Open','High','Low','Close','Volume']].copy()

# =====================================
# 3️⃣ Stop Cluster Detection
# =====================================
lookback = 20

df["recent_high"] = df["High"].rolling(lookback).max()
df["recent_low"] = df["Low"].rolling(lookback).min()

df["stop_cluster_high"] = df["Close"] > df["recent_high"].shift(1)
df["stop_cluster_low"] = df["Close"] < df["recent_low"].shift(1)

# =====================================
# 4️⃣ Institutional Accumulation Zones
# =====================================
# High volume nodes

price_bins = np.linspace(df.Close.min(), df.Close.max(), 80)

df["price_bin"] = pd.cut(df["Close"], price_bins)

volume_profile = df.groupby("price_bin", observed=False)["Volume"].sum()

price_levels = [b.mid for b in volume_profile.index]

volume_profile_df = pd.DataFrame({
    "price": price_levels,
    "volume": volume_profile.values
})

# top liquidity zones
threshold = volume_profile_df["volume"].quantile(0.85)

institutional_zones = volume_profile_df[volume_profile_df["volume"] > threshold]

# =====================================
# 5️⃣ Breakout Liquidity Detection
# =====================================
# detect volatility compression then expansion

df["range"] = df["High"] - df["Low"]

df["range_ma"] = df["range"].rolling(30).mean()

df["compression"] = df["range"] < df["range_ma"] * 0.6

df["breakout"] = (df["range"] > df["range_ma"] * 1.8)

# =====================================
# 6️⃣ VWAP (Institutional Execution)
# =====================================
df["cum_vol"] = df["Volume"].cumsum()
df["cum_vol_price"] = (df["Close"] * df["Volume"]).cumsum()

df["VWAP"] = df["cum_vol_price"] / df["cum_vol"]

# =====================================
# 7️⃣ Optional: Hedge Fund Positioning (COT)
# =====================================
# If you have COT data CSV
# columns expected: Date, ManagedMoneyLong, ManagedMoneyShort

try:
    cot = pd.read_csv("gold_cot.csv")
    cot["Date"] = pd.to_datetime(cot["Date"])

    cot["Net_HedgeFund_Position"] = cot["ManagedMoneyLong"] - cot["ManagedMoneyShort"]

    print("COT positioning loaded")

except:
    print("No COT file detected (skipping hedge fund positioning)")

# =====================================
# 8️⃣ Plot Gold Liquidity Radar
# =====================================
plt.figure(figsize=(16,8))

plt.plot(df.index, df["Close"], label="Gold Price", color="black")

# VWAP
plt.plot(df.index, df["VWAP"], label="VWAP", color="blue")

# Stop clusters
plt.scatter(df[df["stop_cluster_high"]].index,
            df[df["stop_cluster_high"]]["Close"],
            color="red", label="Stop Sweep High")

plt.scatter(df[df["stop_cluster_low"]].index,
            df[df["stop_cluster_low"]]["Close"],
            color="green", label="Stop Sweep Low")

# Breakouts
plt.scatter(df[df["breakout"]].index,
            df[df["breakout"]]["Close"],
            color="purple", label="Breakout Liquidity")

plt.title("Gold Liquidity Radar")
plt.legend()
plt.show()

# =====================================
# 9️⃣ Volume Liquidity Heatmap
# =====================================
plt.figure(figsize=(6,10))

sns.heatmap(
    volume_profile_df[["volume"]],
    yticklabels=np.round(volume_profile_df["price"],2),
    cmap="inferno"
)

plt.title("Gold Institutional Liquidity Heatmap")
plt.xlabel("Volume")
plt.ylabel("Price Level")

plt.show()

# =====================================
# 🔟 Print Institutional Zones
# =====================================
print("\nInstitutional accumulation zones:")
print(institutional_zones)
